In [1]:
!pip install faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.4/31.4 MB 44.3 MB/s eta 0:00:00


In [2]:
import requests, os
from bs4 import BeautifulSoup
import pandas as pd
import numpy as np
import time
import re
from urllib.parse import urljoin, urlparse
from typing import Dict, List, Optional, Tuple
import json
from dataclasses import dataclass, asdict
from difflib import SequenceMatcher
import xml.etree.ElementTree as ET

import google.generativeai as genai


from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [3]:
import logging

# logging.basicConfig(filename='output.logs',level=logging.INFO, force=True)
logging.basicConfig(level=logging.INFO, force=True)
logger = logging.getLogger('logger')


In [4]:

@dataclass
class Product:
    """Product data structure"""
    brand: str
    product_name: str
    model_article_number: str
    category: str
    subcategory: str
    type_id: str
    classification_path: str
    technical_specs: str
    short_description: str
    long_description: str
    product_image_url: str
    datasheet_url: str
    source_url: str

In [5]:
from google.colab import userdata
genai.configure(api_key=userdata.get('GEMINI_API_KEY'))
for m in genai.list_models():
  if 'generateContent' in m.supported_generation_methods:
    print(m.name)

INFO:tornado.access:200 GET /v1beta/models?pageSize=50&%24alt=json%3Benum-encoding%3Dint (127.0.0.1) 6552.05ms


models/gemini-2.5-pro-preview-03-25
models/gemini-2.5-flash-preview-05-20
models/gemini-2.5-flash
models/gemini-2.5-flash-lite-preview-06-17
models/gemini-2.5-pro-preview-05-06
models/gemini-2.5-pro-preview-06-05
models/gemini-2.5-pro
models/gemini-2.0-flash-exp
models/gemini-2.0-flash
models/gemini-2.0-flash-001
models/gemini-2.0-flash-lite-001
models/gemini-2.0-flash-lite
models/gemini-2.0-flash-lite-preview-02-05
models/gemini-2.0-flash-lite-preview
models/gemini-2.0-pro-exp
models/gemini-2.0-pro-exp-02-05
models/gemini-exp-1206
models/gemini-2.0-flash-thinking-exp-01-21
models/gemini-2.0-flash-thinking-exp
models/gemini-2.0-flash-thinking-exp-1219
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/learnlm-2.0-flash-experimental
models/gemma-3-1b-it
models/gemma-3-4b-it
models/gemma-3-12b-it
models/gemma-3-27b-it
models/gemma-3n-e4b-it
models/gemma-3n-e2b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.

INFO:tornado.access:200 GET /v1beta/models?pageSize=50&pageToken=Ch5tb2RlbHMvaW1hZ2VuLTQuMC1nZW5lcmF0ZS0wMDE%3D&%24alt=json%3Benum-encoding%3Dint (127.0.0.1) 637.85ms


In [6]:
"""
Enhanced Gemini API Classifier with Similarity Search
Uses embedding-based similarity to provide only relevant categories to Gemini
"""



class GeminiClassifier:
    """
    Gemini-based classifier with similarity search
    Uses TF-IDF or sentence embeddings to find most relevant categories
    """

    def __init__(self,
                 api_key: str,
                 classification_tree_df: pd.DataFrame,
                 use_sentence_transformers: bool = False):
        """
        Initialize Gemini classifier with similarity search

        Args:
            api_key: Gemini API key
            classification_tree_df: Full classification tree
            use_sentence_transformers: If True, uses sentence-transformers (better but slower)
                                      If False, uses TF-IDF (faster, good enough)
        """
        genai.configure(api_key=api_key)
        self.model = genai.GenerativeModel('gemini-2.5-flash-lite')#gemini-2.5-flash-lite-preview-06-17
        self.tree_df = classification_tree_df

        # Get only Type level (4th level - leaf nodes)
        self.tree_df['depth'] = self.tree_df['path'].str.count('\.') + 1
        self.type_nodes = self.tree_df[self.tree_df['depth'] == 4].copy()

        # Build full text representation for each category
        self.type_nodes['full_text'] = self.type_nodes.apply(
            lambda row: self._build_category_full_text(row), axis=1
        )

        # Initialize similarity search
        self.use_sentence_transformers = use_sentence_transformers
        self._initialize_similarity_search()

    def _build_category_full_text(self, row: pd.Series) -> str:
        """Build complete text representation of a category including parent hierarchy"""
        path_ids = [int(x) for x in row['path'].split('.')]
        names = []

        for pid in path_ids:
            match = self.tree_df[self.tree_df['id'] == pid]
            if not match.empty:
                names.append(match.iloc[0]['name'])

        # Combine: current name + parent names for better matching
        full_text = f"{row['name']} {' '.join(names)}"
        return full_text.lower()

    def _initialize_similarity_search(self):
        """Initialize similarity search mechanism"""
        if self.use_sentence_transformers:
            self._initialize_sentence_transformers()
        else:
            self._initialize_tfidf()

    def _initialize_tfidf(self):
        """Initialize TF-IDF vectorizer (fast, no additional dependencies)"""
        print("Initializing TF-IDF similarity search...")

        # Create TF-IDF vectors for all categories
        self.vectorizer = TfidfVectorizer(
            max_features=500,
            ngram_range=(1, 3),
            stop_words='english',
            lowercase=True
        )

        category_texts = self.type_nodes['full_text'].tolist()
        self.category_vectors = self.vectorizer.fit_transform(category_texts)

        print(f"Indexed {len(category_texts)} categories for similarity search")

    def _initialize_sentence_transformers(self):
        """Initialize sentence transformers (better but requires additional library)"""
        try:
            from sentence_transformers import SentenceTransformer
            print("Initializing Sentence Transformer similarity search...")

            # Use a small, fast model
            self.embedding_model = SentenceTransformer('all-MiniLM-L6-v2')

            # Encode all categories
            category_texts = self.type_nodes['full_text'].tolist()
            self.category_embeddings = self.embedding_model.encode(
                category_texts,
                show_progress_bar=True,
                convert_to_numpy=True
            )

            print(f"Indexed {len(category_texts)} categories with embeddings")

        except ImportError:
            print("Warning: sentence-transformers not installed. Falling back to TF-IDF.")
            print("Install with: pip install sentence-transformers")
            self.use_sentence_transformers = False
            self._initialize_tfidf()

    def _find_similar_categories(self,
                                 product_text: str,
                                 top_k: int = 30) -> pd.DataFrame:
        """
        Find most similar categories using similarity search

        Args:
            product_text: Combined product name, description, specs
            top_k: Number of most similar categories to return

        Returns:
            DataFrame of most relevant categories
        """
        if self.use_sentence_transformers:
            return self._find_similar_with_embeddings(product_text, top_k)
        else:
            return self._find_similar_with_tfidf(product_text, top_k)

    def _find_similar_with_tfidf(self, product_text: str, top_k: int) -> pd.DataFrame:
        """Find similar categories using TF-IDF"""
        # Transform product text to TF-IDF vector
        product_vector = self.vectorizer.transform([product_text.lower()])

        # Calculate cosine similarity
        similarities = cosine_similarity(product_vector, self.category_vectors)[0]

        # Get top-k indices
        top_indices = np.argsort(similarities)[-top_k:][::-1]

        # Return relevant categories with similarity scores
        result = self.type_nodes.iloc[top_indices].copy()
        result['similarity_score'] = similarities[top_indices]

        return result

    def _find_similar_with_embeddings(self, product_text: str, top_k: int) -> pd.DataFrame:
        """Find similar categories using sentence embeddings"""
        # Encode product text
        product_embedding = self.embedding_model.encode(
            product_text.lower(),
            convert_to_numpy=True
        )

        # Calculate cosine similarity
        similarities = cosine_similarity(
            product_embedding.reshape(1, -1),
            self.category_embeddings
        )[0]

        # Get top-k indices
        top_indices = np.argsort(similarities)[-top_k:][::-1]

        # Return relevant categories with similarity scores
        result = self.type_nodes.iloc[top_indices].copy()
        result['similarity_score'] = similarities[top_indices]

        return result

    def _build_category_context(self,
                                product_name: str,
                                description: str,
                                specs: dict,
                                top_k: int = 30) -> str:
        """
        Build category context using similarity search
        Only includes categories most relevant to the product

        Args:
            product_name: Product name
            description: Product description
            specs: Technical specifications
            top_k: Number of most relevant categories to include

        Returns:
            Formatted string of relevant categories
        """
        # Combine product information
        specs_text = ""
        if isinstance(specs, dict):
            specs_text = " ".join([f"{k} {v}" for k, v in specs.items()])
        elif isinstance(specs, str):
            specs_text = specs

        product_text = f"{product_name} {description} {specs_text}"

        # Find most similar categories
        similar_categories = self._find_similar_categories(product_text, top_k)

        # Format for prompt
        context_lines = []
        for idx, row in similar_categories.iterrows():
            full_path = self._get_full_path_names(row['path'])
            score = row['similarity_score']
            context_lines.append(
                f"ID: {row['id']} | Similarity: {score:.3f} | Path: {full_path}"
            )

        return "\n".join(context_lines)

    def _get_full_path_names(self, path: str) -> str:
        """Convert numeric path to human-readable category names"""
        path_ids = [int(x) for x in path.split('.')]
        names = []
        for pid in path_ids:
            match = self.tree_df[self.tree_df['id'] == pid]
            if not match.empty:
                names.append(match.iloc[0]['name'])
        return ' > '.join(names)

    def classify(self,
                 product_name: str,
                 description: str,
                 specs: dict,
                 top_k_categories: int = 30,
                 max_retries: int = 3) -> Tuple[Optional[str], Optional[str], Optional[str]]:
        """
        Classify product using Gemini API with similarity-filtered categories

        Args:
            product_name: Product name
            description: Product description
            specs: Technical specifications
            top_k_categories: Number of most relevant categories to consider
            max_retries: Number of retry attempts

        Returns:
            (type_id, classification_path, full_category_name)
        """

        # Build the prompt with similarity-filtered categories
        prompt = self._build_classification_prompt(
            product_name,
            description,
            specs,
            top_k_categories
        )

        # Call Gemini with retries
        for attempt in range(max_retries):
            try:
                response = self.model.generate_content(prompt)
                result = self._parse_response(response.text)

                if result:
                    return result

                time.sleep(1)  # Rate limiting

            except Exception as e:
                print(f"Gemini API error (attempt {attempt + 1}): {e}")
                if attempt < max_retries - 1:
                    time.sleep(2 ** attempt)  # Exponential backoff
                continue

        return None, None, None

    def _build_classification_prompt(self,
                                     product_name: str,
                                     description: str,
                                     specs: dict,
                                     top_k: int = 30) -> str:
        """Build prompt for Gemini with similarity-filtered categories"""

        # Format specs nicely
        specs_text = ""
        if isinstance(specs, str):
            try:
                specs = json.loads(specs)
            except:
                specs_text = specs

        if isinstance(specs, dict):
            specs_text = "\n".join([f"- {k}: {v}" for k, v in list(specs.items())[:10]])

        # Get relevant categories using similarity search
        category_context = self._build_category_context(
            product_name,
            description,
            specs,
            top_k
        )

        prompt = f"""You are a product classification expert. Classify the following product into the MOST SPECIFIC category from the pre-filtered classification options below.

PRODUCT INFORMATION:
Name: {product_name}
Description: {description}
Technical Specifications:
{specs_text}

MOST RELEVANT CLASSIFICATION OPTIONS:
(These categories were pre-filtered based on similarity to your product)

{category_context}

INSTRUCTIONS:
1. Review the product information carefully
2. Examine the pre-filtered categories (already ranked by relevance)
3. Select the MOST SPECIFIC and ACCURATE match
4. Consider the product's primary function and use case
5. The similarity scores are provided as hints but trust your judgment

RESPONSE FORMAT (JSON only):
{{
    "type_id": "the numeric ID from the classification options",
    "reasoning": "brief explanation of why this category is the best match",
    "confidence": "high|medium|low",
    "similarity_score_used": "the similarity score of the chosen category"
}}

Return ONLY valid JSON, no other text."""

        return prompt

    def _parse_response(self, response_text: str) -> Optional[Tuple[str, str, str]]:
        """Parse Gemini response and validate against classification tree"""
        try:
            # Extract JSON from response
            json_match = re.search(r'\{.*\}', response_text, re.DOTALL)
            if not json_match:
                return None

            result = json.loads(json_match.group())

            type_id = str(result.get('type_id', '')).strip()

            # Validate type_id exists in tree
            match = self.type_nodes[self.type_nodes['id'] == int(type_id)]

            if match.empty:
                print(f"Invalid type_id returned by Gemini: {type_id}")
                return None

            row = match.iloc[0]
            full_path = self._get_full_path_names(row['path'])

            # Log confidence and reasoning
            confidence = result.get('confidence', 'unknown')
            reasoning = result.get('reasoning', 'N/A')
            print(f"  Confidence: {confidence} | Reasoning: {reasoning[:100]}")

            return (
                type_id,
                row['path'],
                full_path
            )

        except Exception as e:
            print(f"Error parsing Gemini response: {e}")
            return None



<>:32: SyntaxWarning: invalid escape sequence '\.'
<>:32: SyntaxWarning: invalid escape sequence '\.'
/tmp/ipython-input-1144613304.py:32: SyntaxWarning: invalid escape sequence '\.'
  self.tree_df['depth'] = self.tree_df['path'].str.count('\.') + 1


In [7]:
from abc import ABC, abstractmethod

class BaseScraper(ABC):
    """
    Base class for web scrapers.

    Child classes must implement the 'scrape_product' and 'get_all_product_urls' methods.
    """

    def __init__(self, url):
        """
        Initialize the scraper with a target URL.

        Args:
            url (str): The URL to scrape.
        """
        self.url = url
        self.data = None

    @abstractmethod
    def scrape_product(self):
        """
        Scrape product data from the given URL.

        This method must be implemented by subclasses.
        """
        raise NotImplementedError("scrape product method is not implemented for this scraper.")


    @abstractmethod
    def get_all_product_urls(self):
        """
        Retrieve all product URLs from the target website.

        This method must be implemented by subclasses.
        """
        raise NotImplementedError("get all product urls method is not implemented for this scraper.")

    def get_product_urls_from_sitemap(self):
        """
        Retrieve product URLs from the website's sitemap.

        This method should be overridden by subclasses if sitemap parsing is supported.
        """
        raise NotImplementedError("Sitemap parsing is not implemented for this scraper.")

In [8]:
class SikaScraper:
    """Scraper for Sika products"""

    def __init__(self):
        self.base_url = "https://gcc.sika.com"
        self.session = requests.Session()
        self.session.headers.update({
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'
        })

    def get_product_urls_from_sitemap(self) -> List[str]:
        """Extract product URLs from sitemap"""
        sitemap_url = f"{self.base_url}/en/products.sitemap.xml"
        logger.info(f"Fetching Sika sitemap: {sitemap_url}")

        try:
            response = self.session.get(sitemap_url, timeout=30)
            response.raise_for_status()

            # Parse XML
            root = ET.fromstring(response.content)

            # Extract URLs (handle namespace)
            namespace = {'ns': 'http://www.sitemaps.org/schemas/sitemap/0.9'}
            urls = [loc.text for loc in root.findall('.//ns:loc', namespace)]
            urls.remove("https://gcc.sika.com/en/products.html")

            logger.info(f"Found {len(urls)} product URLs in sitemap")
            return urls
        except Exception as e:
            logger.error(f"Error fetching sitemap: {e}")
            return []

    def scrape_product(self, url: str) -> Optional[Dict]:
        """Scrape individual Sika product page"""
        try:
            logger.info(f"Scraping Sika product: {url}")

            response = self.session.get(url, timeout=30)
            response.raise_for_status()

            soup = BeautifulSoup(response.content, 'html.parser')

            # Extract product data
            data = {
                'brand': 'Sika',
                'source_url': url
            }

            # Product name
            title_elem = soup.find('h1', {'itemprop': 'name'})
            data['product_name'] = title_elem.text.strip() if title_elem else ''

            # Short description
            short_desc = soup.find('div', class_='cmp-product__description--short')
            data['short_description'] = short_desc.text.strip() if short_desc else ''

            # Long description
            long_desc = soup.find('p', {'itemprop': 'description'})
            data['long_description'] = long_desc.text.strip() if long_desc else ''

            # Extract model/article number from product name or URL
            data['model_article_number'] = self._extract_model_number(data['product_name'], url)

            # Technical specifications
            specs = self._extract_technical_specs(soup)
            data['technical_specs'] = json.dumps(specs)

            # Product image
            img = soup.find('img', {'itemprop': 'image'})
            if not img:
                img = soup.select_one('.cmp-product__image img')
            data['product_image_url'] = urljoin(self.base_url, img['src']) if img and img.get('src') else ''

            # Datasheet URL
            datasheet_link = soup.find('a', href=re.compile(r'\.pdf'))
            data['datasheet_url'] = urljoin(self.base_url, datasheet_link['href']) if datasheet_link else ''

            # Extract usage and advantages for better classification
            usage = soup.find('h3', string='Usage')
            usage_text = usage.find_next('span').text.strip() if usage else ''

            advantages = soup.find('h3', string='Advantages')
            advantages_text = advantages.find_next('span').text.strip() if advantages else ''

            # data['classification_context'] = f"{data['product_name']} {short_desc} {', '.join(f'{k}:{v}' for k, v in specs.items())}"


            return data

        except Exception as e:
            logger.error(f"Error scraping {url}: {e}")
            return None

    def _extract_model_number(self, product_name: str, url: str) -> str:
        """Extract model/article number from product name or URL"""
        # Try to extract from product name
        match = re.search(r'®?-?\s*(\d+[A-Z\d\s-]*)', product_name)
        if match:
            return match.group(1).strip()

        # Fallback: extract from URL
        url_parts = url.rstrip('/').split('/')
        if url_parts:
            return url_parts[-1].replace('.html', '')

        return ''

    def _extract_technical_specs(self, soup: BeautifulSoup) -> Dict:
        """Extract technical specifications from product page"""
        specs = {}

        # Find all specification tables, this logs packaging ...
        # tables = soup.find_all('table')
        # for table in tables:
        #     rows = table.find_all('tr')
        #     for row in rows:
        #         cols = row.find_all('td')
        #         if len(cols) >= 2:
        #             key = cols[0].get_text(strip=True)
        #             value = cols[1].get_text(strip=True)
        #             if key and value:
        #                 specs[key] = value


        # Also extract from h3 sections
        # sections = soup.find_all('h3')
        # for section in sections:
        #   # if section.text == 'Technical data':
        #     next_elem = section.find_next_sibling()
        #     if next_elem and next_elem.name in ['span', 'p', 'table']:
        #         specs[section.text.strip()] = next_elem.get_text(strip=True)[:200]
        # logger.info(specs)
        # Step 1: Find the <a> tag with "Technical Information"
        tech_anchor = soup.find('a', string='Technical Information')

        # Step 2: Get the next <div> that contains the content
        tech_content = None
        if tech_anchor:
          tech_content = tech_anchor.find_next('div', class_='cmp-accordion__content')

        # Step 3: Find all .productcontent blocks inside that section
        if tech_content:
          for block in tech_content.select('.productcontent'):
              title_tag = block.find('h3')
              if not title_tag:
                  continue

              title = title_tag.get_text(strip=True)
              content = block.find('span')

              # Handle <p> values
              if content and content.find('p') and not content.find('table'):
                  values = [p.get_text(strip=True) for p in content.find_all('p')]
                  specs[title] = values if len(values) > 1 else values[0]

              # Handle <table> values
              elif content and content.find('table'):
                  tables = content.find_all('table')
                  table_data = []
                  for table in tables:
                      for row in table.find_all('tr')[1:]:  # skip headers
                          cols = [col.get_text(strip=True) for col in row.find_all('td')]
                          if cols:
                              table_data.append(cols)
                  specs[title] = table_data

        return specs



In [9]:
from urllib.parse import urljoin, urlsplit, urlunsplit

def normalize_url_strip_query(url: str) -> str:
    """Return URL without query string or fragment (keeps scheme://host/path)."""
    if not url:
        return url
    parts = urlsplit(url)
    # rebuild without query and fragment
    normalized = urlunsplit((parts.scheme, parts.netloc, parts.path.rstrip('/'), '', ''))
    return normalized

In [10]:

class FlexToolsScraper:
    """Scraper for Flex-Tools products"""

    def __init__(self):
        self.base_url = "https://www.flex-tools.com"
        self.session = requests.Session()
        self.session.headers.update({
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'
        })

    def get_all_product_urls(self) -> List[str]:
        """Crawl Flex-Tools to get all product URLs"""
        products = []
        main_categories_url = f"{self.base_url}/en-gb/products"

        try:
            # Get main categories
            logger.info(f"Fetching Flex-Tools categories: {main_categories_url}")
            response = self.session.get(main_categories_url, timeout=30)
            response.raise_for_status()

            soup = BeautifulSoup(response.content, 'html.parser')

            # Find all category links
            category_links = soup.select('a.card-link[href*="/products/"]')
            categories = list(set([urljoin(self.base_url, link['href']) for link in category_links if link.get('href')]))

            logger.info(f"Found {len(categories)} main categories")

            # For each category, get subcategories
            for category_url in categories[:10]:  # Limit for testing
                time.sleep(1)  # Be polite
                subcategory_urls = self._get_subcategories(category_url)

                # For each subcategory, get products
                for subcat_url in subcategory_urls:
                    time.sleep(1)
                    product_urls = self._get_products_from_page(subcat_url)

                    # normalize each product url by stripping query/fragments to avoid dupes
                    normalized = [normalize_url_strip_query(u) for u in product_urls]
                    products.extend(normalized)

            logger.info(f"Found {len(products)} total product URLs")
            return list(set(products))  # Remove duplicates

        except Exception as e:
            logger.error(f"Error crawling Flex-Tools: {e}")
            return []

    def _get_subcategories(self, category_url: str) -> List[str]:
        """Get subcategory URLs from a category page"""
        try:
            response = self.session.get(category_url, timeout=30)
            response.raise_for_status()
            soup = BeautifulSoup(response.content, 'html.parser')

            # Find subcategory links
            subcat_links = soup.select('a.card-link[href*="/products/"]')
            subcats = [urljoin(self.base_url, link['href']) for link in subcat_links if link.get('href')]

            # Filter out the current category URL
            subcats = [url for url in subcats if url != category_url]

            return list(set(subcats))
        except Exception as e:
            logger.error(f"Error getting subcategories from {category_url}: {e}")
            return []

    def _get_products_from_page(self, page_url: str) -> List[str]:
        """Extract product URLs from a subcategory page"""
        try:
            response = self.session.get(page_url, timeout=30)
            response.raise_for_status()
            soup = BeautifulSoup(response.content, 'html.parser')

            # Find product cards
            product_cards = soup.select('div.product-card a.card-link')
            products = [urljoin(self.base_url, link['href']) for link in product_cards if link.get('href')]

            return list(set(products))
        except Exception as e:
            logger.error(f"Error getting products from {page_url}: {e}")
            return []
    def _extract_base_data(self, soup: BeautifulSoup, canonical_url: str) -> Dict:
        """
        Extract page-level fields that are common to either the single product or
        every variant (title, descriptions, image, datasheet, base_specs).
        """
        # Title + short description
        title_tag = soup.find('p', class_='product-info-title')
        base_title = title_tag.get_text(strip=True) if title_tag else ''

        short_desc_tag = soup.find('p', class_='product-info-subtitle')
        if not short_desc_tag:
            short_desc_tag = soup.find('p', class_='product-info-description')
        short_desc = short_desc_tag.get_text(strip=True) if short_desc_tag else ''


        # Long description: combine short desc + "Standard supply" section if present
        long_desc_parts = []
        if short_desc:
            long_desc_parts.append(short_desc)
        standard_supply = soup.find('button', string='Standard supply')
        if standard_supply:
            supply_section = standard_supply.find_next('div', class_='collapsible-content')
            if supply_section:
                supply_text = ' '.join([li.get_text(strip=True) for li in supply_section.find_all('li')])
                if supply_text:
                    long_desc_parts.append(f"Standard supply: {supply_text}")
        long_description = '\n'.join(long_desc_parts) if long_desc_parts else short_desc

        # Page-level technical specs (table or dl)
        base_specs = self._extract_specs(soup)

        # Image
        img = soup.select_one('.gallery-main-stage img, .gallery-main img, img.flex-image')
        image_url = ''
        if img and img.get('src'):
            img_src = img['src'].split('?_f')[0] if '?_f' in img['src'] else img['src']
            image_url = urljoin(self.base_url, img_src)

        # Datasheet link (if any)
        link_tag = soup.find('a', class_="download-link", string=lambda text: text and 'Product data sheet' in text)
        if not link_tag:
            for a in soup.select('a.download-link'):
                if a.find('span', string=lambda s: s and 'Product data sheet' in s):
                    link_tag = a
                    break
        data_sheet_url = urljoin(self.base_url, link_tag.get('href')) if link_tag and link_tag.get('href') else ''

        # Classification context for AI processing
        # classification_context = f"{base_title} {short_desc} {', '.join(f'{k}:{v}' for k, v in base_specs.items())}"
        # {long_description}"

        base_data = {
            'brand': 'Flex',
            'product_name': base_title,
            'short_description': short_desc,
            'long_description': long_description,
            'product_image_url': image_url,
            'datasheet_url': data_sheet_url,
            'technical_specs': base_specs,
            'source_url': canonical_url,
            # 'classification_context': classification_context.strip()
        }
        return base_data


    # Variant logic (separated)
    def deal_variant_versions(self, soup: BeautifulSoup, url: str, base_data: Dict) -> Tuple[Optional[List[Dict]], bool]:
        """
        Parse accessory-variant pages. Returns (variants_list, True) when variants detected,
        otherwise (None, False).

        variants_list: list of product dicts that reuse base_data and add variant-specific fields.
        """
        path_lower = urlsplit(url).path.lower()
        has_dummy_akku = 'dummy-akku' in path_lower
        variant_table = soup.select_one('table.accessory-variants-table')

        if not (has_dummy_akku or variant_table):
            return None, False

        variants: List[Dict] = []

        # If there's a structured table, parse rows -> column mapping
        if variant_table:
            thead = variant_table.find('thead')
            tbody = variant_table.find('tbody')
            headers = []
            if thead:
                headers = [th.get_text(strip=True) for th in thead.find_all('th')]
            if not headers:
                # fallback header set (keeps indices stable)
                headers = ["", "Order number", "Battery capacity", "Charge status indicator",
                           "Charge / discharge protection", "Weight", ""]

            if tbody:
                for tr in tbody.find_all('tr'):
                    tds = tr.find_all(['td', 'th'])
                    row_vals = [td.get_text(strip=True) for td in tds]
                    # map headers -> values (by index)
                    specs_map = {}
                    for i, h in enumerate(headers):
                        key = h or f"col_{i}"
                        val = row_vals[i] if i < len(row_vals) else ""
                        specs_map[key] = val

                    # Determine order number: prefer class-based column, fallback to data attr or header
                    order_num = ""
                    article_td = tr.select_one('.column-article-number')
                    if article_td:
                        order_num = article_td.get_text(strip=True)
                    if not order_num and tr.has_attr('data-variant-article'):
                        order_num = tr['data-variant-article']
                    if not order_num:
                        for possible in ('Order number', 'column-article-number', 'Article number'):
                            if possible in specs_map and specs_map[possible]:
                                order_num = specs_map[possible]
                                break

                    # Merge base specs and row-specific specs — row specs override base keys
                    merged_specs = dict(base_data.get('technical_specs') or {})
                    # Normalize keys to strings then update
                    for k, v in specs_map.items():
                        merged_specs[str(k)] = v

                    variant = {
                        'brand': base_data['brand'],
                        'product_name': base_data['product_name'],
                        'model_article_number': order_num or "",
                        'category': '',
                        'subcategory': '',
                        'technical_specs': json.dumps(merged_specs, ensure_ascii=False),
                        'short_description': base_data['short_description'],
                        'long_description': base_data['long_description'],
                        'product_image_url': base_data['product_image_url'],
                        'datasheet_url': base_data['datasheet_url'],
                        'source_url': base_data['source_url'],
                    }
                    variants.append(variant)


        else:
            # No table but dummy-akku-like page: parse the select options to enumerate versions
            select = soup.select_one('select.filter-select-input')
            if select:
                options = select.find_all('option')
                for opt in options:
                    value = opt.get('value', '').strip()
                    label = opt.get_text(strip=True)
                    if not value:
                        continue
                    # Build variant-specific spec entry
                    merged_specs = dict(base_data.get('base_technical_specs') or {})
                    merged_specs['version_label'] = label
                    variant = {
                        'brand': base_data['brand'],
                        'product_name': f"{base_data['product_name']} - {label}",
                        'model_article_number': value,
                        'category': '',
                        'subcategory': '',
                        'technical_specs': json.dumps(merged_specs, ensure_ascii=False),
                        'short_description': base_data['short_description'],
                        'long_description': base_data['long_description'],
                        'product_image_url': base_data['product_image_url'],
                        'datasheet_url': base_data['datasheet_url'],
                        'source_url': base_data['source_url'],
                    }
                    variants.append(variant)

        logger.info(f"Parsed {len(variants)} variants from accessory page: {base_data['source_url']}")
        return (variants if variants else None), bool(variants)


    def scrape_product(self, url: str) -> Optional[Dict]:
      """Scrape individual Flex-Tools product page"""
      try:
          logger.info(f"Scraping Flex-Tools product: {url}")
          response = self.session.get(url, timeout=30)
          response.raise_for_status()

          soup = BeautifulSoup(response.content, 'html.parser')

          data = {
              'brand': 'Flex',
              'source_url': url
          }

          # extract base page-level data (image, descriptions, base specs, datasheet)
          base_data = self._extract_base_data(soup, url)
          data.update(base_data)
          # check variant pages (will reuse base_data)
          variants, is_variant = self.deal_variant_versions(soup, url, base_data)
          if is_variant and variants:
              return variants


          model_article_number = None
          meta_data_elements = soup.find_all('p', class_='product-info-meta-data')

          for elem in meta_data_elements:
              if 'Order number:' in elem.text:
                  model_article_number = elem.text.replace('Order number:', '').strip()
                  break
          try:
            data['model_article_number'] = model_article_number if model_article_number else soup.find('div', class_='details-article-number').get_text(strip=True).replace('Order number:', '').strip() #url.rstrip('/').split('/')[-1]
          except:
            data['model_article_number'] = url.rstrip('/').split('/')[-1]
            logger.warning(url)


          return data

      except Exception as e:
          logger.error(f"Error scraping {url}: {e}")
          return None

    def _extract_specs(self, soup: BeautifulSoup) -> Dict:
      # """Extract technical specifications from Flex Tools product page"""
      # specs = {}

      # try:
      #     # Find the technical data section
      #     tech_data_button = soup.find('button', string='Technical data')
      #     if tech_data_button:
      #         tech_section = tech_data_button.find_next('div', class_='collapsible-content')
      #         if tech_section:
      #             table = tech_section.find('table', class_='technical-data')
      #             if table:
      #                 for row in table.find_all('tr'):
      #                     cells = row.find_all(['th', 'td'])
      #                     if len(cells) == 2:
      #                         key = cells[0].text.strip().lower().replace(' ', '_')
      #                         value = cells[1].text.strip()
      #                         specs[key] = value
      #     logger.info(specs)
      # except Exception as e:
      #     logger.error(f"Error extracting specs: {e}")

      # return specs
        """Extract technical specifications"""
        specs = {}

        # Look for specification tables
        # spec_tables = soup.select('table.specs, .specifications table, .technical-data table')
        # logger.info(spec_tables)
        # for table in spec_tables:
        #     rows = table.find_all('tr')
        #     for row in rows:
        #         cols = row.find_all(['td', 'th'])
        #         if len(cols) >= 2:
        #             key = cols[0].get_text(strip=True)
        #             value = cols[1].get_text(strip=True)
        #             if key and value:
        #                 specs[key] = value
        table = soup.find('table', class_='technical-data')
        if table:
          for row in table.find_all('tr'):
              key = row.find('th').get_text(strip=True)
              value = row.find('td').get_text(strip=True)
              specs[key] = value

        # Look for dl/dt/dd lists
        dl_lists = soup.select('dl.specifications, .specs dl')
        if dl_lists:
          for dl in dl_lists:
              terms = dl.find_all('dt')
              descs = dl.find_all('dd')
              for term, desc in zip(terms, descs):
                  specs[term.get_text(strip=True)] = desc.get_text(strip=True)

        return specs

In [11]:
# def read_urls_from_error_file(file_path='error.txt'):
#   """
#   Reads a file where each line is a URL and returns a list of URLs.

#   Args:
#     file_path: The path to the error file (default is 'error.txt').

#   Returns:
#     A list of URLs read from the file.
#   """
#   urls = []
#   try:
#     with open(file_path, 'r') as f:
#       for line in f:
#         url = line.strip()  # Remove leading/trailing whitespace
#         if url:  # Only add non-empty lines
#           urls.append(url)
#   except FileNotFoundError:
#     print(f"Error: The file '{file_path}' was not found.")
#   except Exception as e:
#     print(f"An error occurred while reading the file: {e}")
#   return urls

# # Example usage (optional - you can remove this if you just want the function definition)
# error_urls = read_urls_from_error_file()
# print(error_urls)
# print(len(error_urls))

In [17]:
class ProductPipeline:
    """Simple pipeline: run scrapers, classify, and always save on exit/error/interrupt."""

    def __init__(self,
                 classification_tree_path: str,
                 output_filename: str = "products_output.csv",
                 skip_from_file:str|None =None):
        GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')#os.getenv('GEMINI_API_KEY')
        tree_df = pd.read_csv(classification_tree_path)

        # Create classifier (user-provided class)
        self.classifier = GeminiClassifier(GEMINI_API_KEY, tree_df, use_sentence_transformers=False)

        self.sika_scraper = SikaScraper()
        self.flex_scraper = FlexToolsScraper()
        self.products = []
        self.output_filename = output_filename
        self.skip_from_file = skip_from_file

    def filter_new_urls(self, existing_csv_path: str, candidate_urls: list[str]) -> list[str]:
        """Filter out URLs that already exist in the CSV file."""
        if existing_csv_path == self.output_filename:
          raise ValueError('the output_filename must be not equal to the existing csv path')
        try:
            products_df = pd.read_csv(existing_csv_path)
            old_source_urls = products_df['source_url'].tolist()
        except FileNotFoundError:
            logger.warning(f"CSV file not found: {existing_csv_path}. Assuming no existing URLs.")
            old_source_urls = []
        if len(old_source_urls) == 0:
          return candidate_urls
        new_urls = [url for url in candidate_urls if url not in old_source_urls]
        logger.info(f"Found {len(new_urls)} new URLs to scrape.")
        return new_urls


    def run(self, max_products_per_site: int = 50, include_flex: bool = True):
        """Run pipeline and ensure results are saved no matter what (finally)."""
        logger.info("Starting Product Extraction Pipeline")


        try:
            # Sika
            logger.info("Scraping Sika products...")
            sika_urls = self.sika_scraper.get_product_urls_from_sitemap()
            # remove later TODO
            if self.skip_from_file:
              sika_urls = self.filter_new_urls(self.skip_from_file, sika_urls)
            for url in sika_urls:
                time.sleep(0.5)
                pdata = self.sika_scraper.scrape_product(url)
                self._handle_scraped(pdata)

            # Flex
            if include_flex:
                logger.info("Scraping Flex-Tools products...")
                flex_urls = self.flex_scraper.get_all_product_urls()
                # remove later TODO
                if self.skip_from_file:
                  flex_urls = self.filter_new_urls(self.skip_from_file, flex_urls)


                for url in flex_urls:
                    time.sleep(0.5)
                    pdata = self.flex_scraper.scrape_product(url)
                    self._handle_scraped(pdata)

            logger.info(f"Pipeline finished: processed {len(self.products)} products")

        except KeyboardInterrupt:
            logger.warning("Pipeline interrupted by user (KeyboardInterrupt). Saving progress...")
        except Exception as e:
            logger.exception("Unhandled exception during pipeline run: %s", e)
        finally:
            # Always save on exit / error / interrupt
            try:
                self.export_to_csv(self.output_filename)
            except Exception as e:
                logger.exception("Failed to save output in finally block: %s", e)

    def _handle_scraped(self, product_data: Optional[Dict]):
        """Normalize scraper output (None, dict, or list[dict]) and process."""
        if not product_data:
            return
        if isinstance(product_data, list):
            for item in product_data:
                p = self._process_product_safe(item)
                if p:
                    self.products.append(p)
        else:
            p = self._process_product_safe(product_data)
            if p:
                self.products.append(p)

    def _process_product_safe(self, data: Dict):
        """Wrap actual processing to avoid a single product crashing the run."""
        try:
            return self._process_product(data)
        except Exception as e:
            logger.exception("Error processing product '%s': %s", data.get('product_name'), e)
            # store minimal failed record for traceability
            self.products.append({
                'brand': data.get('brand', ''),
                'product_name': data.get('product_name', ''),
                'model_article_number': data.get('model_article_number', ''),
                'type_id': '',
                'classification_path': '',
                'source_url': data.get('source_url', ''),
                'error': str(e)
            })
            return None

    def _process_product(self, data: Dict):
        """Classify and return a Product dataclass (or dict if dataclass construction fails)."""
        raw = self.classifier.classify(
            data.get('product_name', ''),
            data.get('short_description', ''),
            data.get('technical_specs', '')
        )

        # normalize classifier output (tuple or dict)
        type_id = ''
        class_path = ''
        readable = ''
        if raw is None:
            pass
        elif isinstance(raw, (tuple, list)):
            # expecting (type_id, classification_path, readable_path)
            try:
                type_id, class_path, readable = raw
            except Exception:
                # tolerant fallback
                type_id = str(raw[0]) if len(raw) > 0 else ''
                class_path = raw[1] if len(raw) > 1 else ''
        elif isinstance(raw, dict):
            type_id = str(raw.get('type_id') or '')
            class_path = raw.get('classification_path') or raw.get('path') or ''
            readable = raw.get('full_label') or raw.get('readable_path') or ''
        else:
            # attribute-style object?
            type_id = str(getattr(raw, 'type_id', '') or '')
            class_path = getattr(raw, 'classification_path', '') or ''

        if not readable and class_path and hasattr(self.classifier, 'df'):
            # simple resolution: map ids in path to names
            try:
                ids = [int(x) for x in str(class_path).split('.') if x]
                names = []
                for pid in ids:
                    r = self.classifier.df[self.classifier.df['id'] == pid]
                    if not r.empty:
                        names.append(str(r.iloc[0]['name']))
                readable = " > ".join(names)
            except Exception:
                readable = ''

        # Build Product dataclass (fallback to dict)
        try:
            prod = Product(
                brand=data.get('brand', ''),
                product_name=data.get('product_name', ''),
                model_article_number=data.get('model_article_number', ''),
                category=(readable.split(' > ')[1] if readable and '>' in readable else ''),
                subcategory=(readable.split(' > ')[2] if readable and readable.count('>') >= 2 else ''),
                type_id=type_id,
                classification_path=class_path,
                technical_specs=data.get('technical_specs', ''),
                short_description=data.get('short_description', ''),
                long_description=data.get('long_description', ''),
                product_image_url=data.get('product_image_url', ''),
                datasheet_url=data.get('datasheet_url', ''),
                source_url=data.get('source_url', '')
            )
            return prod
        except Exception as e:
            logger.exception("Failed to build Product dataclass: %s", e)
            return {
                'brand': data.get('brand', ''),
                'product_name': data.get('product_name', ''),
                'model_article_number': data.get('model_article_number', ''),
                'category': (readable.split(' > ')[1] if readable and '>' in readable else ''),
                'subcategory': (readable.split(' > ')[2] if readable and readable.count('>') >= 2 else ''),
                'type_id': type_id,
                'classification_path': class_path,
                'technical_specs': data.get('technical_specs', ''),
                'short_description': data.get('short_description', ''),
                'long_description': data.get('long_description', ''),
                'product_image_url': data.get('product_image_url', ''),
                'datasheet_url': data.get('datasheet_url', ''),
                'source_url': data.get('source_url', '')
            }

    def export_to_csv(self, filename: str = None):
        """Save products (dataclass or dict) to CSV."""
        fname = filename or self.output_filename
        rows = []
        for p in self.products:
            if hasattr(p, "__dataclass_fields__"):
                rows.append(asdict(p))
            elif isinstance(p, dict):
                rows.append(p)
            else:
                try:
                    rows.append(dict(p))
                except Exception:
                    logger.debug("Skipping unknown product type: %s", type(p))
        df = pd.DataFrame(rows)
        df.to_csv(fname, index=False)
        logger.info("Exported %d products to %s", len(rows), fname)


In [13]:
def main():
    """Main entry point"""
    # Path to classification tree
    CLASSIFICATION_TREE = 'classification_tree.csv'

    # Initialize and run pipeline
    pipeline = ProductPipeline(CLASSIFICATION_TREE)

    # Run with limited products for testing (remove limit for full run)
    pipeline.run() # include_flex=False

    # Export results its now done automaticcally in finally
    # pipeline.export_to_csv('products_output.csv')


main()


INFO:logger:Starting Product Extraction Pipeline
INFO:logger:Scraping Flex-Tools products...
INFO:logger:Fetching Flex-Tools categories: https://www.flex-tools.com/en-gb/products


Initializing TF-IDF similarity search...
Indexed 354 categories for similarity search


INFO:logger:Found 17 main categories
INFO:logger:Found 223 total product URLs
INFO:logger:Scraping Flex-Tools product: https://www.flex-tools.com/en-gb/products/drills-and-screwdrivers/drywall-screwdriver/dw-37-12-ec-c
INFO:tornado.access:200 POST /v1beta/models/gemini-2.5-flash-lite:generateContent?%24alt=json%3Benum-encoding%3Dint (127.0.0.1) 3897.29ms


  Confidence: high | Reasoning: The product is a cordless drywall screwdriver with a high no-load speed, indicative of a tool design


INFO:logger:Scraping Flex-Tools product: https://www.flex-tools.com/en-gb/products/metal-surface-finishing/trinoxflex-basic-motors/bme-18-0-ec-c
INFO:tornado.access:200 POST /v1beta/models/gemini-2.5-flash-lite:generateContent?%24alt=json%3Benum-encoding%3Dint (127.0.0.1) 1467.71ms


...

  Confidence: high | Reasoning: The product is a cordless vacuum cleaner designed for plant equipment, capable of handling both wet 


INFO:logger:Scraping Flex-Tools product: https://www.flex-tools.com/en-gb/products/cordless-tools/10-8-v-12-v/mwl-150-4
INFO:tornado.access:200 POST /v1beta/models/gemini-2.5-flash-lite:generateContent?%24alt=json%3Benum-encoding%3Dint (127.0.0.1) 1341.48ms


  Confidence: low | Reasoning: The product is a multi-purpose headlamp, which falls under the broad category of 'Special purpose el


INFO:logger:Scraping Flex-Tools product: https://www.flex-tools.com/en-gb/products/cordless-tools/18-0-v/ose-2-80-18-ec-c
INFO:tornado.access:200 POST /v1beta/models/gemini-2.5-flash-lite:generateContent?%24alt=json%3Benum-encoding%3Dint (127.0.0.1) 1214.88ms
INFO:logger:Pipeline finished: processed 223 products
INFO:logger:Exported 223 products to products_output.csv


  Confidence: high | Reasoning: The product is described as a 'cordless orbital sander' and the 'Random orbital disc sander (recharg


In [19]:
import pandas as pd

df = pd.read_csv('products_output.csv')

print("--- Head of products_output.csv ---")
display(df.head())

print("\n--- Tail of products_output.csv ---")
display(df.tail())

--- Head of products_output.csv ---


,brand,product_name,model_article_number,category,subcategory,type_id,classification_path,technical_specs,short_description,long_description,product_image_url,datasheet_url,source_url
0,Sika,SikaWrap®-600 C WV,600 C WV,Construction chemicals,Construction chemicals (other),119523.0,118814.119445.119522.119523,"{""Laminate Nominal Thickness"": ""0.331 mm"", ""La...","Woven unidirectional carbon fibre fabric, desi...",SikaWrap®-600 C WV is a unidirectional woven h...,NaN,https://gcc.sika.com/content/dam/dms/gcc/k/We%...,https://gcc.sika.com/en/construction/refurbish...
1,Sika,Sika AnchorFix®-2+ Tropical,2,Electric tool,Grinder (electrical),116817.0,115547.116749.116816.116817,"{""Compressive Strength"": [], ""Modulus of Elast...",Anchoring adhesive for medium to high loads,"Solvent and styrene free, epoxy acrylate based...",NaN,https://gcc.sika.com/content/dam/dms/gcc/k/We%...,https://gcc.sika.com/en/construction/refurbish...
2,Sika,SikaGrout®-212 AE,212 AE,Construction chemicals,"Joint sealant, water stop, joint profile (buil...",119483.0,118814.119445.119482.119483,"{""Compressive Strength"": [[""W/P ratio = 0.14 (...","CEMENTITIOUS, SHRINKAGE COMPENSATED, NON-SHRIN...","SikaGrout®-212 AE is a cementitious, free flow...",NaN,https://gcc.sika.com/content/dam/dms/gcc/k/We%...,https://gcc.sika.com/en/construction/refurbish...
3,Sika,SikaGrout®-275,275,Construction chemicals,"Joint sealant, water stop, joint profile (buil...",119492.0,118814.119445.119482.119492,"{""Compressive Strength"": [[""W/P = 0.12 (3.0 L ...","Cementitious, high strength, non-shrink, preci...","SikaGrout®-275 is a cementitious, pre-bagged, ...",NaN,https://gcc.sika.com/content/dam/dms/gcc/k/We%...,https://gcc.sika.com/en/construction/refurbish...
4,Sika,Sikadur®-42 PC,42 PC,Construction chemicals,"Joint sealant, water stop, joint profile (buil...",119483.0,118814.119445.119482.119483,"{""Compressive Strength"": [[""1 d"", ""~65 N/mm2""]...","3-PART, EPOXY GROUTING SYSTEM FOR PILE CAPPING","Sikadur®-42 PC is a three-component, moisture ...",NaN,https://gcc.sika.com/content/dam/dms/gcc/k/We%...,https://gcc.sika.com/en/construction/refurbish...



--- Tail of products_output.csv ---


,brand,product_name,model_article_number,category,subcategory,type_id,classification_path,technical_specs,short_description,long_description,product_image_url,datasheet_url,source_url
1090,Flex,LB 125 18.0-EC C,499307,Electric tool,Grinder (rechargeable),116909.0,115547.116749.116906.116909,"{'Battery voltage': '18 V', 'Battery capacity'...","Cordless angle grinder with brake 18.0 V, 125 mm","Cordless angle grinder with brake 18.0 V, 125 mm",https://www.flex-tools.com/_next/image?url=htt...,https://hub.flex-tools.com/_f/185/185b803db429...,https://www.flex-tools.com/en-gb/products/cord...
1091,Flex,FHE 1-16 18.0-EC C,531311,Electric tool,Hammer drill (rechargeable),116863.0,115547.116749.116862.116863,"{'Battery voltage': '18 V', 'Battery capacity'...",Compact and light cordless rotary hammer drill...,Compact and light cordless rotary hammer drill...,https://www.flex-tools.com/_next/image?url=htt...,https://hub.flex-tools.com/_f/f4d/f4de7e10ba13...,https://www.flex-tools.com/en-gb/products/cord...
1092,Flex,VCE 22 L MC 2x18.0,531637,Electric tool,Vacuum Cleaner (Plant Equipment),158363.0,115547.116749.158362.158363,"{'Battery voltage': '18 V', 'Battery capacity'...",Cordless safety vacuum cleaner with manual fil...,Cordless safety vacuum cleaner with manual fil...,https://www.flex-tools.com/_next/image?url=htt...,https://hub.flex-tools.com/_f/b42/b424ff797378...,https://www.flex-tools.com/en-gb/products/vacu...
1093,Flex,MWL 150 4,534476,Electric tool,Special purpose electric tool (electrical),116976.0,115547.116749.116944.116976,"{'Battery voltage': '4 V', 'Colour temperature...",Multi-purpose headlamp 4 V,Multi-purpose headlamp 4 V,https://www.flex-tools.com/_next/image?url=htt...,https://hub.flex-tools.com/_f/474/474601e94990...,https://www.flex-tools.com/en-gb/products/cord...
1094,Flex,OSE 2-80 18-EC C,532084,Electric tool,Grinder (rechargeable),116910.0,115547.116749.116906.116910,"{'Battery voltage': '18 V', 'Battery capacity'...",18 V cordless orbital sander with speed control,18 V cordless orbital sander with speed control,https://www.flex-tools.com/_next/image?url=htt...,https://hub.flex-tools.com/_f/f2d/f2d56ad16a11...,https://www.flex-tools.com/en-gb/products/cord...


In [20]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1095 entries, 0 to 1094
Data columns (total 13 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   brand                 1095 non-null   object 
 1   product_name          1095 non-null   object 
 2   model_article_number  1095 non-null   object 
 3   category              1077 non-null   object 
 4   subcategory           1077 non-null   object 
 5   type_id               1077 non-null   float64
 6   classification_path   1077 non-null   object 
 7   technical_specs       1095 non-null   object 
 8   short_description     1095 non-null   object 
 9   long_description      1095 non-null   object 
 10  product_image_url     223 non-null    object 
 11  datasheet_url         1068 non-null   object 
 12  source_url            1095 non-null   object 
dtypes: float64(1), object(12)
memory usage: 111.3+ KB
